# Emulator GP vs. bezpośrednie próbkowanie mastera EMsoftu

Porównanie profili eCHORD z emulatora (`gp_echord`) z „prawdą”, czyli z `echord_cone.cone_profiles`
(próbkowanie master patternu EMsoftu wzdłuż stożka) dla **losowych orientacji spoza zbioru uczącego**,
w geometrii pomiaru: ω = 0, 2, …, 358°.

Porównanie ma sens tylko przy tych samych parametrach, na których model był uczony
(kąt stożka, `invert`, zbieżność) — notebook bierze je z metadanych modelu.

In [ ]:

import numpy as np, gp_echord as G
ang = G.uniform_canonical_design(16384, n_fold=4, seed=0)      # (Phi, phi2) [rad] dla C4; 2^14 punktów
eu = np.column_stack([np.zeros(len(ang)), np.degrees(ang)])   # phi1 = 0

with open("design_C4_sobol16k.txt", "w") as f:
    f.write(f"eu\n{len(eu)}\n")
    np.savetxt(f, eu, fmt="%.8f")

Uczenie procesu Gaussowskiego na wygenerowanych wyzej kątach

In [ ]:
# import sys
# import train_gp

# sys.argv = [
#     "train_gp.py",
#     "--master",
#     "../2026_10_19_example_PZ/masterKTN_014_01.h5",
#     "--angles",
#     "design_C4_sobol16k.txt",
#     "--theta",
#     "40",
#     "--dpsi",
#     "0.25",
#     "--out",
#     "gp_ktn_th40.npz",
# ]

# train_gp.main()

In [ ]:
# --- ustawienia ---------------------------------------------------------
import os


DATA_DIR = "/home/pz/workspace/agh/gaussian/2026_10_19_example_PZ"
CUR_DIR = os.getcwd()

MASTER = os.path.join(DATA_DIR, "masterKTN_014_01.h5")
ANGLES = os.path.join(CUR_DIR, "design_C4_sobol16k.txt") # używane tylko, jeśli modelu jeszcze nie ma
TIF_PATH = os.path.join(DATA_DIR, "KTN_014_s2_fin_crop_372x372_bin_62x62.tif") 

MODEL  = os.path.join(CUR_DIR, "gp_ktn_th40.npz")          # wynik train_gp.py

PHI0, SENSE = 0.0, +1               # geometria pomiaru (do kalibracji)
N_TEST, SEED = 300, 1

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import echord_cone as ec
import gp_echord as G
import gp_echord_gpytorch as T

torch = 1

mp = ec.MasterPattern(MASTER)
if os.path.exists(MODEL) and not torch:
    print(f"wczytuje model z {MODEL}")
    gp = G.EchordGP.load(MODEL)
    print(f"wczytano model z {MODEL}")
elif torch:
    print(f"uczenie modelu TORCH z {MASTER} ...")
    ops = G.proper_rotations(G.proper_group_name(mp.space_group))
    data = G.build_training_data(mp, anglefile=ANGLES, coneAngle=40, dpsi=0.25)
    gp = T.TorchEchordGP(ops).fit(data, n_fit=2000)
    gp.save("model.pt")
else:  # pierwszy raz: uczenie (kilka minut), potem model jest wczytywany z pliku
    print(f"uczenie modelu z {MASTER} ...")
    ops = G.proper_rotations(G.proper_group_name(mp.space_group))
    data = G.build_training_data(mp, anglefile=ANGLES, coneAngle=40, dpsi=0.25)
    gp = G.EchordGP(ops).fit(data, n_fit=2000)
    gp.save(MODEL)


meta = gp.meta
print(gp.describe())
print(f"uczony na {len(gp.X_train)} orientacjach; stożek {meta['coneAngle']}°, "
      f"invert={meta['invert']}, zbieżność {meta['sigma_conv_deg']}°")

## Predykcja vs. symulacja dla losowych orientacji

In [ ]:
q = G.random_quaternions(N_TEST, SEED)          # jednorodnie na SO(3)

t0 = time.perf_counter()
omega, P_gp, sd = gp.predict_profiles(q, phi0=PHI0, sense=SENSE)
t_gp = time.perf_counter() - t0

t0 = time.perf_counter()
_, P_sim = mp.cone_profiles(q, meta["coneAngle"], 2.0, phi0=PHI0, sense=SENSE,
                            invert=meta["invert"], sigma_conv_deg=meta["sigma_conv_deg"])
t_sim = time.perf_counter() - t0

r = G.rowwise_pearson(P_gp, P_sim)                       # podobieństwo kształtu profilu
rmse = np.sqrt(np.mean((P_gp - P_sim) ** 2, axis=1))
z = (P_sim - P_gp) / sd[:, None]                          # błąd w jednostkach niepewności GP
eu = G.matrix_to_bunge(ec.quat_to_matrix(q))
dist, _ = gp.kernel.min_misorientation(G.canonical_matrices(eu[:, 1:]), gp.X_train)

print(f"Pearson r: mediana {np.median(r):.3f}, 10% {np.percentile(r, 10):.3f}, min {r.min():.3f}")
print(f"RMSE: mediana {np.median(rmse):.3f} (zakres intensywności {np.ptp(P_sim):.2f})")
print(f"kalibracja: sd(z) = {z.std():.2f} (ideał 1), |z| < 2: {100 * np.mean(np.abs(z) < 2):.1f}% (ideał ~95%)")
print(f"czas dla {N_TEST} profili: emulator {t_gp:.2f} s, bezpośrednio {t_sim:.2f} s")

### Przykłady: najlepszy, typowy (mediana) i najgorszy profil
Szary pas to ±2σ emulatora; `odl.` to kąt dezorientacji do najbliższej orientacji uczącej.

In [ ]:
order = np.argsort(r)
picks = [("najlepszy", order[-1]), ("mediana", order[len(order) // 2]), ("najgorszy", order[0])]
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, (label, i) in zip(axes, picks):
    ax.fill_between(omega, P_gp[i] - 2 * sd[i], P_gp[i] + 2 * sd[i], color="0.85", label="GP ±2σ")
    ax.plot(omega, P_sim[i], "k.-", ms=3, lw=0.8, label="cone_profiles (master EMsoft)")
    ax.plot(omega, P_gp[i], "-", color="tab:red", lw=1.2, label="emulator GP")
    ax.set_title(f"{label}: r = {r[i]:.3f}, odl. = {dist[i]:.2f}°", fontsize=10)
    ax.set_ylabel("intensywność")
axes[0].legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("ω [°]")
plt.tight_layout()

### Podsumowanie dla wszystkich orientacji testowych
Jakość zależy przede wszystkim od odległości do najbliższego punktu uczącego — to mówi, czy
zbiór uczący jest wystarczająco gęsty. Prawy panel: jeśli niepewność jest dobrze skalibrowana,
histogram z pokrywa się z krzywą N(0, 1).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].hist(r, bins=30, color="tab:blue")
ax[0].set_xlabel("Pearson r (GP vs. symulacja)"); ax[0].set_ylabel("liczba profili")

ax[1].scatter(dist, r, s=8, alpha=0.6)
ax[1].set_xlabel("odl. do najbliższej orientacji uczącej [°]"); ax[1].set_ylabel("Pearson r")

x = np.linspace(-5, 5, 200)
ax[2].hist(np.clip(z.ravel(), -5, 5), bins=60, density=True, color="tab:green", alpha=0.7)
ax[2].plot(x, np.exp(-x**2 / 2) / np.sqrt(2 * np.pi), "k", label="N(0, 1)")
ax[2].set_xlabel("z = (symulacja − GP) / σ"); ax[2].legend()
plt.tight_layout()

### Drobna struktura: profil kanoniczny na gęstej siatce
Pomiar co 2° ukrywa szpilki. Tu porównanie dla typowej orientacji na siatce emulatora
(krok `dpsi`), w oknie wokół najwyższego piku.

In [ ]:
i = picks[1][1]
Xc = G.canonical_matrices(eu[i:i + 1, 1:])
F_gp, F_sd = gp.predict_canonical(Xc)
F_sim = G.simulate_canonical(mp, eu[i:i + 1, 1:], meta["coneAngle"], gp.dpsi,
                             meta["invert"], meta["sigma_conv_deg"])
psi = np.arange(F_gp.shape[1]) * gp.dpsi
c = psi[np.argmax(F_sim[0])]
win = np.abs((psi - c + 180) % 360 - 180) < 30
s = np.argsort((psi[win] - c + 180) % 360 - 180)
x = ((psi[win] - c + 180) % 360 - 180)[s]
plt.figure(figsize=(11, 3.2))
plt.fill_between(x, (F_gp[0] - 2 * F_sd[0])[win][s], (F_gp[0] + 2 * F_sd[0])[win][s], color="0.85")
plt.plot(x, F_sim[0][win][s], "k-", lw=1, label="symulacja")
plt.plot(x, F_gp[0][win][s], "-", color="tab:red", lw=1.2, label="emulator GP")
plt.xlabel("ψ − ψ(pik) [°]"); plt.ylabel("intensywność"); plt.legend(fontsize=8)
plt.title(f"profil kanoniczny, krok {gp.dpsi}°, r(całość) = {G.rowwise_pearson(F_gp, F_sim)[0]:.3f}", fontsize=10)
plt.tight_layout()

Model z GPyTorch ma ten sam interfejs, więc wystarczy w trzeciej komórce podmienić wczytywanie na
`import gp_echord_gpytorch as T; gp = T.TorchEchordGP.load("model.pt")` — reszta notebooka działa bez zmian.